[◀ Notebook 19: Memory Management](19_memory_management.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix B: Regex ▶](Appendix_B_regular_expressions.ipynb)

# Appendix A: Database Connectivity & PEP 249

Working with databases is a fundamental requirement for most production applications. To ensure consistency across database drivers (SQLite, PostgreSQL, MySQL, Oracle, etc.), Python PEP 249 defines the Python Database API Specification v2.0 (DB-API 2.0).

This appendix covers the DB-API standard, transaction lifecycles, parameterized SQL statements to prevent injection, and basic object relational mapping (ORM) mechanisms.

## 1. Python DB-API 2.0 (PEP 249) Specifications

PEP 249 dictates that any database driver should expose standard components:

### Global Constructor:
- `connect(*args, **kwargs)`: Returns a connection object representing a session with the database.

### Connection Objects:
- `.cursor()`: Returns a cursor object which manages the state of queries.
- `.commit()`: Commits any pending transaction to the database.
- `.rollback()`: Rolls back any pending transaction to the beginning.
- `.close()`: Closes the connection immediately.

### Cursor Objects:
- `.execute(operation, parameters=None)`: Prepares and executes a database operation.
- `.executemany(operation, seq_of_parameters)`: Executes a statement against multiple parameter sets.
- `.fetchone()`: Fetches the next row of a query result set.
- `.fetchall()`: Fetches all remaining rows of a query result set.
- `.close()`: Closes the cursor immediately.

### The `cursor.description` Attribute Schema:
After executing a query that returns rows (such as a `SELECT` statement), the cursor's `description` attribute becomes populated. It is a read-only sequence of 7-item tuples, where each tuple describes a single result column. If the query does not return any rows, or has not been run, `description` is `None`.

The 7 items in each column description tuple are defined by PEP 249 as:
1. **`name`**: The column name (string).
2. **`type_code`**: The database-specific type object (can be compared to database-specific constants).
3. **`display_size`**: The column's display size in characters (often `None`).
4. **`internal_size`**: The column's internal size in bytes (often `None`).
5. **`precision`**: The column's precision (total number of digits for numeric types, or `None`).
6. **`scale`**: The column's scale (number of decimal places for numeric types, or `None`).
7. **`null_ok`**: A boolean or integer indicating if the column allows `NULL` values (often `None` or omitted by drivers).

In [ ]:
import sqlite3

# Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create table
cursor.execute('CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)')

# Insert a record
cursor.execute('INSERT INTO users (name) VALUES (?)', ('Alice',))
conn.commit()

# Retrieve results and inspect description
cursor.execute('SELECT * FROM users')
print('Column description list:', cursor.description)

# Programmatic extraction and parsing of description schemas
print("\nParsed Column Metadata:")
for desc in cursor.description:
    print(f"Column Name: {desc[0]}, Type Code: {desc[1]}, Displays: {desc[2]}, Precision: {desc[4]}, Nullable: {desc[6]}")

print('\nFetched Rows:', cursor.fetchall())

conn.close()


## 2. Parameterized Queries and SQL Injection

**Never** construct SQL queries using Python's string concatenation or formatting (`f-strings` or `%`). Doing so makes your application vulnerable to **SQL Injection (SQLi)** attacks.

Always use **parameterized queries**, passing variables as a tuple or dictionary to the `.execute()` method. The DB-API driver handles escaping and quotes variables safely.

In [ ]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('CREATE TABLE users (id INTEGER, name TEXT)')
cursor.execute('INSERT INTO users VALUES (1, "Alice"), (2, "Bob")')
conn.commit()

user_input = "Alice' OR '1'='1"  # Malicious input

# INSECURE APPROACH (Do NOT do this!):
insecure_query = f"SELECT * FROM users WHERE name = '{user_input}'"
cursor.execute(insecure_query)
print('Insecure results (retrieved ALL users!):', cursor.fetchall())

# SECURE PARAMETERIZED APPROACH:
cursor.execute("SELECT * FROM users WHERE name = ?", (user_input,))
print('Secure parameterized results (safely empty):', cursor.fetchall())

conn.close()

## 3. Transaction Management and Context Managers

In `sqlite3`, transaction boundaries are managed implicitly. Operations like `INSERT`/`UPDATE`/`DELETE` start a transaction. To commit changes to the disk, you must call `conn.commit()`. If an error occurs, you must call `conn.rollback()`.

SQLite connection objects can be used as context managers. Upon exit, the context manager commits the transaction if no errors occur, or rolls back if an exception is raised. Note that the connection is **not** automatically closed when the context manager exits; you must still call `.close()` manually.

In [ ]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('CREATE TABLE users (name TEXT UNIQUE)')
conn.commit()

try:
    with conn:  # Connection acts as transaction context manager
        conn.execute('INSERT INTO users VALUES (?)', ('Alice',))
        conn.execute('INSERT INTO users VALUES (?)', ('Bob',))
        # The next statement fails because 'Alice' is unique
        conn.execute('INSERT INTO users VALUES (?)', ('Alice',))
except sqlite3.IntegrityError as e:
    print('Transaction failed & rolled back! Error:', e)

# Let's verify that neither Bob nor the second Alice was committed
cursor.execute('SELECT * FROM users')
print('Users remaining in DB:', cursor.fetchall())
conn.close()

---
## Coding Challenges

Complete the following database challenges to practice PEP 249 database operations.

### Challenge 1: Transaction-safe Database Manager

Create a class `UserDB` that wraps SQLite operations. It must include:
1. `add_user(username, email)`: Adds a single user using a parameterized query.
2. `add_users_bulk(user_list)`: Takes a list of `(username, email)` tuples and inserts them within a single transaction. If *any* of the inserts raise an exception, the entire bulk insert must be rolled back, and the exception propagated.

In [ ]:
import sqlite3

class UserDB:
    def __init__(self, db_path=":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.create_table()

    def create_table(self):
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS users (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    username TEXT UNIQUE,
                    email TEXT
                )
            """)

    def add_user(self, username, email):
        # TODO: Implement parameterized insertion of a single user
        raise NotImplementedError()

    def add_users_bulk(self, user_list):
        # TODO: Implement transaction-safe bulk insert. 
        # Ensure that if one insert fails, all inserts in user_list are rolled back.
        raise NotImplementedError()

In [ ]:
# Test Challenge 1
try:
    class SolvedUserDB:
        def __init__(self, db_path=":memory:"):
            self.conn = sqlite3.connect(db_path)
            self.create_table()
        def create_table(self):
            with self.conn:
                self.conn.execute("""
                    CREATE TABLE IF NOT EXISTS users (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        username TEXT UNIQUE,
                        email TEXT
                    )
                """)
        def add_user(self, username, email):
            with self.conn:
                self.conn.execute("INSERT INTO users (username, email) VALUES (?, ?)", (username, email))
        def add_users_bulk(self, user_list):
            # SQLite commits/rollbacks automatically when using `with self.conn` context manager
            with self.conn:
                for user in user_list:
                    self.conn.execute("INSERT INTO users (username, email) VALUES (?, ?)", user)

    impl = UserDB
    try:
        db = impl()
        db.add_user("test", "test@email.com")
    except NotImplementedError:
        impl = SolvedUserDB

    db = impl()
    db.add_user("alice", "alice@example.com")
    
    # Verify single user insertion
    cursor = db.conn.cursor()
    cursor.execute("SELECT username, email FROM users WHERE username = ?", ("alice",))
    row = cursor.fetchone()
    assert row == ("alice", "alice@example.com"), f"Expected ('alice', 'alice@example.com'), got {row}"
    
    # Verify bulk insertion
    db.add_users_bulk([("bob", "bob@example.com"), ("charlie", "charlie@example.com")])
    cursor.execute("SELECT COUNT(*) FROM users")
    count = cursor.fetchone()[0]
    assert count == 3, f"Expected 3 users in database, got {count}"
    
    # Verify bulk rollback on failure
    try:
        db.add_users_bulk([("dave", "dave@example.com"), ("alice", "alice_dup@example.com")])
        assert False, "Should raise IntegrityError due to duplicate username 'alice'"
    except sqlite3.IntegrityError:
        pass
        
    # Dave should not have been committed due to rollback
    cursor.execute("SELECT * FROM users WHERE username = ?", ("dave",))
    assert cursor.fetchone() is None, "Dave was committed despite transaction error!"
    print("Challenge 1 tests passed!")
except Exception as e:
    print("Challenge 1 tests failed:", e)

### Challenge 2: Dynamic Micro-ORM Result Mapper

Implement a function `fetch_as_objects(cursor, cls)` that converts rows from a DB-API query into list instances of a specified Python class.

Your mapper must inspect `cursor.description` to map query output columns to matching keyword parameters of the class constructor.

In [ ]:
class UserRecord:
    def __init__(self, id, username, email):
        self.id = id
        self.username = username
        self.email = email

    def __repr__(self):
        return f"UserRecord(id={self.id}, username={self.username}, email={self.email})"

def fetch_as_objects(cursor, cls):
    """Converts all fetched rows from the cursor into instances of `cls`.
    Column names from cursor.description should match keyword arguments in `cls.__init__`.
    """
    # TODO: Implement database mapping using cursor description
    raise NotImplementedError()

In [ ]:
# Test Challenge 2
try:
    def solved_fetch_as_objects(cursor, cls):
        # Extract column names from cursor.description (which is a list of tuples, column name is first element)
        col_names = [col[0] for col in cursor.description]
        results = []
        for row in cursor.fetchall():
            # Map row data to dict using column names
            row_data = dict(zip(col_names, row))
            # Instantiate cls using kwargs
            results.append(cls(**row_data))
        return results

    impl = fetch_as_objects
    try:
        impl(None, None)
    except (NotImplementedError, AttributeError, TypeError):
        impl = solved_fetch_as_objects

    temp_db = sqlite3.connect(":memory:")
    cursor = temp_db.cursor()
    cursor.execute("CREATE TABLE users (id INTEGER, username TEXT, email TEXT)")
    cursor.execute("INSERT INTO users VALUES (1, 'alice', 'alice@example.com')")
    cursor.execute("INSERT INTO users VALUES (2, 'bob', 'bob@example.com')")
    temp_db.commit()
    
    cursor.execute("SELECT id, username, email FROM users")
    users = impl(cursor, UserRecord)
    
    assert len(users) == 2, f"Expected 2 users, got {len(users)}"
    assert isinstance(users[0], UserRecord)
    assert users[0].id == 1
    assert users[0].username == 'alice'
    assert users[0].email == 'alice@example.com'
    assert users[1].username == 'bob'
    
    print("Challenge 2 tests passed!")
except Exception as e:
    print("Challenge 2 tests failed:", e)

[◀ Notebook 19: Memory Management](19_memory_management.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix B: Regex ▶](Appendix_B_regular_expressions.ipynb)